# Gold Output Validation

Lightweight validation for Gold v1 analytical outputs. Gold v1 is useful for inspection, but it is not a final accounting model or dimensional mart.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "src" / "processing").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "processing").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / "projects" / "03-retail-revenue-analytics" / "src" / "processing").exists():
    PROJECT_ROOT = cwd / "projects" / "03-retail-revenue-analytics"
else:
    raise RuntimeError("Launch this notebook from the project directory, notebook directory, or repository root.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from processing.gold.metadata import get_gold_output_dir, get_gold_metadata_dir

gold_output_dir = get_gold_output_dir()
gold_metadata_dir = get_gold_metadata_dir()
gold_output_dir, gold_metadata_dir

## Available Gold Outputs

In [ ]:
gold_files = sorted(gold_output_dir.glob("*.csv"))
gold_files

In [ ]:
import pandas as pd

outputs = {path.stem: pd.read_csv(path) for path in gold_files}
{name: dataframe.shape for name, dataframe in outputs.items()}

## Output Previews

In [ ]:
for name, dataframe in outputs.items():
    print(f"\n{name}: {dataframe.shape[0]} rows x {dataframe.shape[1]} columns")
    display(dataframe.head())

## Revenue Sanity Checks

These checks compare Gold outputs against each other. They do not replace formal data quality tests.

In [ ]:
kpis = outputs.get("kpi_overview")
status = outputs.get("order_status_summary")

if kpis is not None and status is not None:
    kpi_gmv = float(kpis.loc[kpis["metric_name"] == "gross_merchandise_value", "metric_value"].iloc[0])
    status_gmv = round(float(status["gross_merchandise_value"].sum()), 2)
    print({"kpi_gmv": kpi_gmv, "status_summary_gmv": status_gmv, "matches": kpi_gmv == status_gmv})

In [ ]:
for name, dataframe in outputs.items():
    print(f"\n{name}")
    print(list(dataframe.columns))
    display(dataframe.isna().sum().sort_values(ascending=False).head(10))